# Trabalho Prático 1: Encontrando Exoplanetas com Machine Learning

#### Aluno: *Pedro Dias Soares*

## Tópicos
### 1. Introdução e Entendimento do Problema
Nesta seção, contextualizaremos o uso do telescópio espacial Kepler e o desafio astrofísico de diferenciar a assinatura luminosa de um exoplaneta genuíno das anomalias instrumentais conhecidas como falsos positivos.

#### 1.1 Dicionário de Dados
Aqui detalharemos o significado das principais variáveis observadas (como profundidade do trânsito, temperatura da estrela, etc.), com destaque para a nossa variável alvo categórica: `koi_disposition`.

---

### 2. Importação dos Dados
Carga do arquivo `koi_data.csv` para o ambiente de desenvolvimento e visualização inicial das primeiras linhas para checagem de integridade.

---

### 3. Prevenção de Data Leakage e Separação de Dados
O conjunto de dados será dividido antes de qualquer transformação: 70% para a base de treinamento e 30% separados (hold-out) para o teste final.

---

### 4. Limpeza e Tratamento de Dados Ausentes
Tratamento de valores nulos e inconsistências sintáticas/semânticas. Cálculos de preenchimento utilizarão estatísticas do conjunto de treinamento.

---

### 5. Seleção de Atributos
Análise exploratória para remover colunas irrelevantes ou que causem a maldição da dimensionalidade. As escolhas de exclusão serão justificadas com base no domínio do problema.

---

### 6. Transformação de Dados
Aplicação de técnicas de normalização.

---

### 7. Balanceamento de Classes
Estratégias para lidar com a diferença de volume entre as classes de exoplanetas confirmados e falsos positivos, aplicadas de forma isolada no conjunto de treino.

---

### 8. Treinamento, Validação Cruzada e Ajuste de Hiperparâmetros
Implementação do dicionário de modelos contendo os seis algoritmos requisitados: 
1. Naive Bayes;
2. Decision Tree;
3. k-Nearest Neighbors (k-NN);
4. Support Vector Machines (SVM);
5. Random Forest;
6. Gradient Tree Boosting;

Os ajustes serão validados via Cross-Validation usando apenas os 70% de treino.

---

### 9. Avaliação no Conjunto de Teste
Aplicação dos modelos otimizados sobre os 30% de dados intocados para simular o comportamento algorítmico no mundo real.

---

### 10. Resultados e Análise Crítica
#### 10.1 Métricas e Matrizes de Confusão
Extração das métricas vitais para o domínio astrofísico, com foco na Sensibilidade (Recall) e na convergência da Equação harmônica do F1-Score através das Matrizes de Confusão.

#### 10.2 Conclusão
Comparativo de desempenho entre as arquiteturas, interpretando o real impacto das decisões de pré-processamento nos resultados e atestando a corretude conceitual do experimento.

---

### 11. Referencial Teórico

## 1. Introdução e Entendimento do Problema

A base de dados empírica utilizada (`koi_data.csv`) provém do telescópio espacial Kepler, uma missão da NASA desenhada para explorar a estrutura e a diversidade de sistemas planetários. O Kepler revolucionou a astronomia ao identificar candidatos a planetas através do **Método de Trânsito** (Borucki et al., 2010). Este método consiste em monitorar continuamente a curva de luz de milhares de estrelas; quando um planeta orbita e cruza a linha de visão do telescópio, ocorre uma micro-queda periódica e detectável no brilho estelar.

### O Desafio dos Falsos Positivos na Astrofísica

Apesar da precisão do método de trânsito, a simples detecção de uma queda de luminosidade não confirma a existência de um exoplaneta. Inúmeros fenômenos astrofísicos e instrumentais mimetizam perfeitamente a assinatura fotométrica de um trânsito real, gerando o que a literatura chama de **Falsos Positivos**. Segundo Morton et al. (2016), as principais fontes de confusão incluem:
* **Sistemas Binários Eclipsantes:** Estrelas companheiras que bloqueiam a luz mutuamente.
* **Manchas Estelares e Variabilidade:** Alterações na própria superfície da estrela hospedeira.
* **Ruído Instrumental:** Anomalias térmicas ou de sensores no próprio telescópio.

### A Abordagem de Aprendizado de Máquina

Historicamente, a triagem (vetting) desses sinais exigia análise humana exaustiva. Com o aumento exponencial do volume de dados, a adoção de técnicas de inteligência artificial tornou-se o padrão-ouro na astronomia moderna (Shallue & Vanderburg, 2018). 

Neste cenário, modelamos o problema como uma tarefa de **Aprendizado de Máquina Supervisionado**. O objetivo deste trabalho é aplicar algoritmos de classificação multicritério sobre atributos físicos e geométricos do trânsito como profundidade da curva, período orbital e temperatura estelar para distinguir automaticamente os exoplanetas da classe `CONFIRMED` das anomalias da classe `FALSE POSITIVE`. O modelo que melhor generalizar esses padrões matemáticos servirá como um filtro autônomo e confiável para futuras missões espaciais.

O objetivo é treinar e comparar múltiplos algoritmos de classificação contendo os seis algoritmos requisitados: 

1. Naive Bayes;
2. Decision Tree;
3. k-Nearest Neighbors (k-NN);
4. Support Vector Machines (SVM);
5. Random Forest;
6. Gradient Tree Boosting; 

---



### 1.1 Dicionário de Dados e Seleção Preliminar

Para que os algoritmos de aprendizado de máquina consigam discernir padrões astrofísicos reais de anomalias, é essencial compreender o significado físico das features coletadas pelo telescópio. O conjunto `koi_data.csv` é extraído do catálogo Kepler Objects of Interest (KOI). De acordo com o Dicionário de Dados Oficial do NASA Exoplanet Archive (2021), as variáveis mais relevantes para a modelagem dividem-se nas seguintes categorias:

#### Variável Alvo
* **`koi_disposition`**: O status de classificação oficial do objeto de interesse. No nosso escopo de classificação binária, esta coluna contém os valores `CONFIRMED` (Exoplaneta confirmado por outras metodologias e publicações) ou `FALSE POSITIVE` (Alarme falso causado por binárias eclipsantes, ruído instrumental, etc.).

#### Características do Trânsito
Estas variáveis descrevem o evento físico da queda de luz captada pelo telescópio.
* **`koi_period`**: O período orbital do candidato, ou seja, o tempo em dias que o objeto leva para completar uma volta ao redor de sua estrela.
* **`koi_duration`**: A duração total do trânsito em horas. Representa o tempo exato em que a luz da estrela ficou bloqueada.
* **`koi_depth`**: A profundidade do trânsito medida em partes por milhão. É a métrica crucial da curva de luz, indicando a fração exata de luminosidade bloqueada pelo objeto.
* **`koi_prad`**: O raio estimado do corpo planetário (em raios terrestres), calculado com base na profundidade do trânsito e no tamanho da estrela.

#### Características da Estrela Hospedeira
O ambiente estelar interfere diretamente na probabilidade de habitabilidade e na validação do planeta.
* **`koi_steff`**: A temperatura efetiva da estrela hospedeira (em Kelvin). Estrelas muito quentes ou muito frias geram curvas de luz com assinaturas de ruído diferentes.
* **`koi_srad`**: O raio fotossférico da estrela (em raios solares). Um trânsito de mesma profundidade em uma estrela gigante implica um objeto muito maior do que em uma estrela anã.
* **`koi_smass`**: A massa da estrela (em massas solares). Fundamental para calcular a força gravitacional e a mecânica orbital do sistema.

## Referencial Teórico

* Borucki, W. J., et al. (2010). Kepler Planet-Detection Mission: Introduction and First Results. *Science*, 327(5968), 977-980.
* Morton, T. D., et al. (2016). False Positive Probabilities for all Kepler Objects of Interest: 1284 Newly Confirmed Planets and 428 Likely False Positives. *The Astrophysical Journal*, 822(2), 86.
* Shallue, C. J., & Vanderburg, A. (2018). Identifying Exoplanets with Deep Learning: A Five-planet Resonant Chain around Kepler-80 and an Eighth Planet around Kepler-90. *The Astronomical Journal*, 155(2), 94.